# English emotion classification (dair-ai/emotion, 6 classes)

**Source:** Part 1 of the original master notebook (cells 3-18) plus the English half of section S9 (cells 76-78).

## 1. Objective and experimental scope
Classify English sentences into six emotions (`sadness, joy, love, anger, fear, surprise`) and compare
classical TF-IDF models, word-embedding recurrent networks and a fine-tuned DistilBERT.

| Block | What the original notebook did |
|---|---|
| Data | `dair-ai/emotion`, configuration `unsplit` (416,809 rows) → exact-duplicate removal on `text` → 393,822 rows |
| Split | stratified **70 / 15 / 15** (`random_state=42`) |
| Classical | TF-IDF (20k features, 1-2 grams, sublinear) + LogReg / LinearSVM / MultinomialNB / SGD |
| Recurrent | LSTM, GRU, BiLSTM (5 epochs, no validation use) |
| Transformer | `distilbert-base-uncased`, "light CPU" config: 8,000 training rows, 1 epoch |
| Analysis | confusion matrix + misclassified examples + one LIME explanation (LogReg) |
| Secondary | S9: a *different* 80/20 LogReg/LinearSVM pipeline on the raw text |

> **Status of this notebook.** The code below is the refactored version and calls `src/`. It has **not** been re-executed
> in the environment used to prepare the repository (no dataset download, no GPU/PyTorch there). The numbers quoted in
> §8 are those *recorded in the original notebook's outputs* (see `results/tables/*__src-cell*.csv`).

In [ ]:
import sys
from pathlib import Path

# Locate the repository root (works from the repo root or from notebooks/) and make `src` importable.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "__init__.py").is_file())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils import get_paths, get_device, set_seed, save_table, save_figure

paths = get_paths(create=True)      # data/, results/, cache/, checkpoints/ (override with ITC_* env vars)
SEED = 42
set_seed(SEED)                      # seed once, as the original notebook did
DEVICE = get_device()
print("repo root:", ROOT, "| device:", DEVICE)

## 2. Dataset loading and validation

In [ ]:
from src.data import load_dair_emotion, DAIR_LABEL_NAMES

label_names = DAIR_LABEL_NAMES                       # id 0..5, validated against the Hub features in the loader
df_dair = load_dair_emotion(cache_dir=paths.cache_dir, add_label_name=False)

print("--- Initial DataFrame ---")
print("Shape:", df_dair.shape)
print("Missing values:\n", df_dair.isnull().sum())
print("Duplicates:", df_dair.duplicated(subset=["text"]).sum())

df_dair = df_dair.drop_duplicates(subset=["text"]).reset_index(drop=True)   # exact duplicates only
df_dair["emotion"] = df_dair["label"].map(lambda x: label_names[x])
print("Shape after dropping duplicates:", df_dair.shape)
print(df_dair["emotion"].value_counts())

## 3. Preprocessing and feature preparation
`clean_text_dair`: lower-case, replace everything outside `[a-z0-9'\s]` by a space, collapse whitespace.
Split: 30 % held out (stratified) then halved into validation and test.

In [ ]:
from src.preprocessing import clean_text_dair
from src.data import split_train_val_test_via_temp

df_dair["text_clean"] = df_dair["text"].apply(clean_text_dair)
df_dair.to_csv(paths.data_dir / "english_dair_ai_cleaned.csv", index=False)   # derived data, git-ignored

X_train_raw, X_val_raw, X_test_raw, y_train, y_val, y_test = split_train_val_test_via_temp(
    df_dair["text_clean"], df_dair["label"], seed=SEED
)
print(f"Train: {len(X_train_raw)} | Val: {len(X_val_raw)} | Test: {len(X_test_raw)}")

In [ ]:
# EDA: class distribution and text length (words)
df_dair["text_len"] = df_dair["text_clean"].str.split().str.len()
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.countplot(data=df_dair, x="emotion", order=df_dair["emotion"].value_counts().index,
              hue="emotion", legend=False, ax=axes[0], palette="viridis")
axes[0].set_title("Class Distribution"); axes[0].tick_params(axis="x", rotation=45)
sns.histplot(df_dair["text_len"], bins=50, ax=axes[1], color="skyblue", kde=True)
axes[1].set_title("Text Length Distribution"); axes[1].set_xlabel("Number of Words")
plt.tight_layout()
save_figure(fig, paths.figures_dir, "english_emotion", "eda_class_distribution_and_length")
plt.show()
print(df_dair["text_len"].describe())

## 4. Model configuration
* TF-IDF: `DAIR_TFIDF_CFG` (`max_features=20000, ngram_range=(1,2), sublinear_tf=True`), **fitted on train only**.
* Classical models: `dair_classical_models(SEED)` (class-balanced except Naive Bayes).
* Recurrent models: `DAIR_RNN` (vocab 10,000, max length 64, embedding 128, hidden 64, dropout 0.3, Adam 1e-3, 5 epochs, batch 128).
* DistilBERT: `DAIR_DISTILBERT_CPU` (lr 5e-5, batch 4, 1 epoch, first 8,000 rows of each split, max length 64).

All hyper-parameters are copied from the original cells and verified against them by `tests/test_equivalence_original.py`.

## 5. Training / inference - classical models (evaluated on the test split)

In [ ]:
import time
from src.features import DAIR_TFIDF_CFG, fit_transform_tfidf
from src.models import dair_classical_models, fit_predict_timed
from src.evaluation import compute_classification_metrics, text_classification_report

t0 = time.time()
vectorizer, X_train_tfidf, (X_val_tfidf, X_test_tfidf) = fit_transform_tfidf(
    X_train_raw, [X_val_raw, X_test_raw], **DAIR_TFIDF_CFG
)
print(f"TF-IDF fit+transform: {time.time()-t0:.1f}s | shape {X_train_tfidf.shape}")

classical_models = dair_classical_models(SEED)
results_list = []
for name, clf in classical_models.items():
    print(f"\nTraining {name}...")
    y_pred, t_train, t_inf = fit_predict_timed(clf, X_train_tfidf, y_train, X_test_tfidf)
    res = compute_classification_metrics(y_test, y_pred, name.split(" + ")[1], "TF-IDF", t_train, t_inf)
    results_list.append(res)
    print(f"Accuracy: {res['Accuracy']:.4f} | Macro F1: {res['Macro F1']:.4f}")
    print(text_classification_report(y_test, y_pred, label_names))

### Recurrent models
> **Caveat preserved from the original (not fixed):** sequences are right-padded and the network reads the hidden state of the
> *last position* (`out[:, -1, :]`), i.e. after the padding tokens for any text shorter than 64 words. See `docs/methodology.md` §7.
> Any correction would be a methodological change and needs your approval.

In [ ]:
import torch.nn as nn, torch
from src.preprocessing import build_vocab, encode_texts
from src.preprocessing.sequences import embedding_vocab_size
from src.models.rnn import DAIR_RNN, RNNClassifier, make_loader, train_fixed_epochs, predict_classes

cfg = DAIR_RNN
vocab = build_vocab(X_train_raw, cfg["vocab_size"])
X_train_dl = encode_texts(X_train_raw, vocab, cfg["max_len"])
X_test_dl = encode_texts(X_test_raw, vocab, cfg["max_len"])
train_loader = make_loader(X_train_dl, y_train.values, cfg["batch_size"], shuffle=True)
test_loader = make_loader(X_test_dl, y_test.values, cfg["eval_batch_size"])

for rnn_type in cfg["rnn_types"]:
    print(f"\nTraining {rnn_type}...")
    model = RNNClassifier(rnn_type, embedding_vocab_size(vocab), cfg["embed_dim"], cfg["hidden_dim"],
                          num_outputs=6, dropout=cfg["dropout"]).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"])
    _, t_train = train_fixed_epochs(model, train_loader, nn.CrossEntropyLoss(), optimizer, cfg["epochs"], DEVICE)
    preds, t_inf = predict_classes(model, test_loader, DEVICE)
    res = compute_classification_metrics(y_test, preds, rnn_type, "Word Embeddings", t_train, t_inf)
    results_list.append(res)
    print(f"Accuracy: {res['Accuracy']:.4f} | Macro F1: {res['Macro F1']:.4f}")

### DistilBERT ("light CPU" configuration)
Trained on the **first 8,000 rows** of the training split; validation = first 8,000 validation rows (checkpoint selection by
`load_best_model_at_end`); metrics are computed on the **first 8,000 test rows only** (not the 59,074-row test split), so this row is
*not directly comparable* with the other rows of the table.

In [ ]:
from src.models.hf_finetuning import DAIR_DISTILBERT_CPU, accuracy_macro_f1_metrics, fine_tune_sequence_classifier

b = DAIR_DISTILBERT_CPU
n = b["max_samples"]
out = fine_tune_sequence_classifier(
    model_name=b["model_name"],
    train_texts=X_train_raw.iloc[:n].tolist(), train_labels=y_train.iloc[:n].tolist(),
    val_texts=X_val_raw.iloc[:n].tolist(), val_labels=y_val.iloc[:n].tolist(),
    test_texts=X_test_raw.iloc[:n].tolist(), test_labels=y_test.iloc[:n].tolist(),
    num_labels=6, training_kwargs=b["training_kwargs"],
    output_dir=str(paths.checkpoints_dir / "dair_distilbert_cpu"),
    max_length=b["max_length"], compute_metrics=accuracy_macro_f1_metrics,
    label_column="labels", use_padding_collator=True, cpu=b["cpu"],
)
res = compute_classification_metrics(y_test.iloc[:n], out["y_pred"], "DistilBERT CPU", b["model_name"],
                                     out["train_time"], out["inference_time"])
results_list.append(res)
print(f"Accuracy: {res['Accuracy']:.4f} | Macro F1: {res['Macro F1']:.4f}")
print(text_classification_report(y_test.iloc[:n], out["y_pred"], label_names))

## 6. Evaluation - final comparison

In [ ]:
from src.evaluation import DAIR_RESULT_COLUMNS

df_results = pd.DataFrame(results_list)[DAIR_RESULT_COLUMNS]
display(df_results)
save_table(df_results, paths.tables_dir, "english_emotion", "model_comparison", split="test")

## 7. Visualisation and error analysis (Logistic Regression, test split)
The original analysed the **logistic regression**, chosen as "best classical model" in the notebook text; in the recorded run the
Linear SVM scored a higher macro-F1 (see §8).

In [ ]:
from src.evaluation import plot_confusion_matrix, misclassified_examples

best_clf = classical_models["TF-IDF + Logistic Regression"]
y_pred_best = best_clf.predict(X_test_tfidf)

fig = plot_confusion_matrix(y_test, y_pred_best, label_names, "Confusion Matrix - Logistic Regression")
save_figure(fig, paths.figures_dir, "english_emotion", "confusion_matrix", model="logistic-regression", split="test")
plt.show()

wrong = np.where(y_test.values != y_pred_best)[0]
print(f"Total misclassified samples: {len(wrong)} out of {len(y_test)}")
for ex in misclassified_examples(X_test_raw.tolist(), y_test.values, y_pred_best, label_names, n=5):
    print(f"\nText: {ex['text']}\nActual: {ex['actual']} | Predicted: {ex['predicted']}")

### LIME explanation (single instance, qualitative)
First misclassified test sample, explained for the model's top predicted class. This illustrates the model's behaviour on one example;
it does not measure explanation quality. Requires `pip install lime`.

In [ ]:
from sklearn.pipeline import make_pipeline
from src.evaluation.explain import explain_instance

pipeline_en = make_pipeline(vectorizer, best_clf)
idx = int(wrong[0]) if len(wrong) else 42
text_instance = X_test_raw.iloc[idx]
print("Text:", text_instance, "\nActual:", label_names[y_test.iloc[idx]])
exp, weights, top_label = explain_instance(pipeline_en.predict_proba, text_instance, label_names, num_features=8, top_labels=1)
print("Model predicted:", label_names[top_label])
for word, w in weights:
    print(f"  - {word:15s} : {w:+.4f}")
fig = exp.as_pyplot_figure(label=top_label); fig.set_size_inches(8, 4); plt.tight_layout()
save_figure(fig, paths.figures_dir, "english_emotion", "lime_explanation", model="logistic-regression", split="test")
plt.show()

## Secondary experiment - S9 English pipeline (different protocol)
Original cells 77-78. **Not comparable with Part 1:** raw (not `clean_text_dair`) text, de-duplication on `(text, emotion)` (416,123 rows),
**80/20** stratified split without validation set, TF-IDF with 50k features / `min_df=2` / `max_df=0.95`, and models without a seed.

In [ ]:
from sklearn.preprocessing import LabelEncoder
from src.data import split_train_test
from src.features import EMOTION_PIPELINE_TFIDF_CFG
from src.models import emotion_pipeline_models
from src.preprocessing import prepare_emotion
from src.evaluation import compute_emotion_pipeline_metrics

english_raw = load_dair_emotion(cache_dir=paths.cache_dir)           # un-deduplicated, with 'emotion' column
emotion_en = prepare_emotion(english_raw, "text")
le_en = LabelEncoder(); y_en = le_en.fit_transform(emotion_en["emotion"])
X_tr, X_te, y_tr, y_te = split_train_test(emotion_en["text"], y_en, test_size=0.2, seed=SEED)
vec9, Xtr9, (Xte9,) = fit_transform_tfidf(X_tr, [X_te], **EMOTION_PIPELINE_TFIDF_CFG)

rows = []
for name, clf in emotion_pipeline_models().items():
    clf.fit(Xtr9, y_tr); pred = clf.predict(Xte9)
    rows.append(compute_emotion_pipeline_metrics(y_te, pred, name, "EN"))
    print(f"\n=== Classification report EN ({name}) ===")
    print(text_classification_report(y_te, pred, le_en.classes_))
res_en_s9 = pd.DataFrame(rows); display(res_en_s9)
save_table(res_en_s9, paths.tables_dir, "english_emotion", "s9_pipeline_80_20", split="test")

## 8. Interpretation (based only on results recorded in the original run)
Recorded in the original outputs (`results/tables/english_emotion__model_comparison__test__src-cell12.csv`), test split:

| Model | Accuracy | Macro-F1 |
|---|---|---|
| Logistic Regression | 0.9366 | 0.9124 |
| Linear SVM | 0.9428 | 0.9174 |
| Naive Bayes | 0.8647 | 0.7868 |
| SGD Classifier | 0.9280 | 0.9018 |
| LSTM | 0.3494 | 0.1356 |
| GRU | 0.9633 | 0.9381 |
| BiLSTM | 0.4146 | 0.2170 |
| DistilBERT CPU (8k train, 8k test rows) | 0.9449 | 0.9218 |

* Among the classical models the Linear SVM has the highest macro-F1; Naive Bayes has high precision but low recall on `love` and `surprise`.
* The GRU has the best figures of the table, while the LSTM and BiLSTM, with the same architecture otherwise, are far lower.
  One plausible explanation is the right-padding / last-position issue described above; **this was not tested**, so it remains a hypothesis.
* DistilBERT was trained on 8,000 rows and scored on 8,000 test rows; it should not be ranked against rows scored on 59,074 test rows.
* Logistic-regression errors: 3,747 of 59,074 test samples. The single LIME example (a `sadness` text predicted `love`) is driven mostly by the word "sympathetic".
* S9 (different protocol, 80/20): LogReg accuracy 0.9052 / macro-F1 0.8720; LinearSVM 0.8924 / 0.8448. The ranking of the two models is the reverse of Part 1,
  which shows that the conclusions depend on preprocessing, split and TF-IDF settings.
* Single run, single seed: no confidence intervals or significance tests were computed.